# 序列逆置 （加注意力的seq2seq）
使用attentive sequence to sequence 模型将一个字符串序列逆置。例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个加attentino的sequence to sequence 模型示意图)
![attentive seq2seq](./seq2seq-attn.jpg)

In [6]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [7]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['QILJVUXLBH', 'JPQHLZLZWX'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[17,  9, 12, 10, 22, 21, 24, 12,  2,  8],
       [10, 16, 17,  8, 12, 26, 12, 26, 23, 24]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0,  8,  2, 12, 24, 21, 22, 10, 12,  9],
       [ 0, 24, 23, 26, 12, 26, 12,  8, 17, 16]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 8,  2, 12, 24, 21, 22, 10, 12,  9, 17],
       [24, 23, 26, 12, 26, 12,  8, 17, 16, 10]])>)


# 建立sequence to sequence 模型

完成两空，模型搭建以及单步解码逻辑

In [8]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz=27
        self.hidden = 128
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64, 
                                                    batch_input_shape=[None, None])
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
                                           return_sequences=True, return_state=True)
        self.dense_attn = tf.keras.layers.Dense(self.hidden)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
        
    @tf.function
    def call(self, enc_ids, dec_ids):
        '''
        完成带attention机制的 sequence2sequence 模型的搭建
        '''
        # 1. 编码器部分
        enc_emb = self.embed_layer(enc_ids)  # (b_sz, enc_len, 64)
        enc_out, enc_state = self.encoder(enc_emb)  # enc_out: (b_sz, enc_len, 128), enc_state: (b_sz, 128)
        
        # 2. 解码器部分
        dec_emb = self.embed_layer(dec_ids)  # (b_sz, dec_len, 64)
        
        # 使用RNN层一次性处理所有时间步
        # 但为了注意力，我们需要每个时间步的输出和状态
        # 这里我们使用RNN并返回序列
        dec_out_all, dec_state = self.decoder(dec_emb, initial_state=enc_state)  # dec_out_all: (b_sz, dec_len, 128)
        
        # 3. 计算注意力
        # 双线性注意力: 对于每个解码器时间步，计算与所有编码器时间步的注意力权重
        
        # 扩展维度以便广播
        # enc_out: (b_sz, enc_len, hidden)
        # dec_out_all: (b_sz, dec_len, hidden)
        
        # 计算注意力分数
        # 方法：将解码器输出投影后与编码器输出做点积
        # 首先投影解码器输出
        dec_proj = self.dense_attn(dec_out_all)  # (b_sz, dec_len, hidden)
        
        # 计算注意力分数: (b_sz, dec_len, enc_len)
        # 使用矩阵乘法: dec_proj (b_sz, dec_len, hidden) 与 enc_out (b_sz, enc_len, hidden) 的转置
        attention_scores = tf.matmul(dec_proj, enc_out, transpose_b=True)  # (b_sz, dec_len, enc_len)
        
        # 应用softmax得到注意力权重
        attention_weights = tf.nn.softmax(attention_scores, axis=-1)  # (b_sz, dec_len, enc_len)
        
        # 计算上下文向量: 加权求和
        # attention_weights: (b_sz, dec_len, enc_len)
        # enc_out: (b_sz, enc_len, hidden)
        context = tf.matmul(attention_weights, enc_out)  # (b_sz, dec_len, hidden)
        
        # 4. 结合解码器输出和上下文向量
        combined = tf.concat([dec_out_all, context], axis=-1)  # (b_sz, dec_len, 2*hidden)
        
        # 5. 输出层
        logits = self.dense(combined)  # (b_sz, dec_len, v_sz)
        
        return logits
    
    
    @tf.function
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids) # shape(b_sz, len, emb_sz)
        enc_out, enc_state = self.encoder(enc_emb)
        # 返回编码器输出和状态
        return enc_out, [enc_state]
    
    @tf.function
    def get_next_token(self, x, state, enc_out):
        '''
        单步解码，用于预测阶段
        x: 当前输入的token，shape (b_sz,)
        state: 解码器状态 [dec_state]
        enc_out: 编码器输出 (b_sz, enc_len, hidden)
        '''
        # 1. 词嵌入
        x = tf.expand_dims(x, axis=1)  # (b_sz, 1)
        inp_emb = self.embed_layer(x)  # (b_sz, 1, 64)
        
        # 2. 解码器单步
        # decoder_cell 接受 (batch, features) 和状态
        dec_out, new_state = self.decoder_cell(inp_emb[:, 0, :], state[0])  # dec_out: (b_sz, hidden)
        
        # 3. 计算注意力
        # 将解码器输出投影
        dec_proj = self.dense_attn(dec_out)  # (b_sz, hidden)
        dec_proj = tf.expand_dims(dec_proj, axis=1)  # (b_sz, 1, hidden)
        
        # 计算注意力分数
        attention_scores = tf.matmul(dec_proj, enc_out, transpose_b=True)  # (b_sz, 1, enc_len)
        attention_scores = tf.squeeze(attention_scores, axis=1)  # (b_sz, enc_len)
        attention_weights = tf.nn.softmax(attention_scores, axis=-1)  # (b_sz, enc_len)
        
        # 计算上下文向量
        context = tf.reduce_sum(tf.expand_dims(attention_weights, axis=-1) * enc_out, axis=1)  # (b_sz, hidden)
        
        # 4. 结合解码器输出和上下文
        combined = tf.concat([dec_out, context], axis=-1)  # (b_sz, 2*hidden)
        
        # 5. 输出层
        logits = self.dense(combined)  # (b_sz, v_sz)
        
        # 6. 预测下一个token
        out = tf.argmax(logits, axis=-1)  # (b_sz,)
        
        # 7. 返回新的状态（格式与encode返回的状态一致）
        new_state_list = [new_state]
        return out, new_state_list

# Loss函数以及训练逻辑

In [9]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

@tf.function
def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(2000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [10]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
train(model, optimizer, seqlen=20)

step 0 : loss 3.3004901
step 500 : loss 1.6088102
step 1000 : loss 0.38513938
step 1500 : loss 0.0991238


<tf.Tensor: shape=(), dtype=float32, numpy=0.036763623>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [11]:
def sequence_reversal():
    def decode(init_state, steps, enc_out):
        b_sz = tf.shape(init_state[0])[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state, enc_out)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 20)
    enc_out, state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1], enc_out), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[True, False, True, True, True, True, True, True, True, True, False, True, True, True, True, True, True, False, True, True, True, True, True, True, True, True, False, True, True, True, True, True]
[('WZNPBWPPSYWJNAMELDWS', 'SWDLEMANJWYSPPWBPNZW'), ('NXLQOUMTXSCSCTWEOGKY', 'YKGOEWTCSCSXTMUOQLXN'), ('HOZFOHQRRPRCRFMLOHZP', 'PZHOLMFRCRPRRQHOFZOA'), ('EUUZCKKENUUMFXOLKSSU', 'USSKLOXFMUUNEKKCZUUE'), ('JJBHMZZYULSGKWICFADP', 'PDAFCIWKGSLUYZZMHBJJ'), ('UCANYAMJQDQFREEWXPZU', 'UZPXWEERFQDQJMAYNACU'), ('UPBFHBNIPCJCQVFQWCDF', 'FDCWQFVQCJCPINBHFBPU'), ('FKHNWIFNCGTWAFSULTCJ', 'JCTLUSFAWTGCNFIWNHKF'), ('CFWOOSFSDXVXTLWDKKHV', 'VHKKDWLTXVXDSFSOOWFC'), ('LDYGORHCVOGPOFLTKJKY', 'YKJKTLFOPGOVCHROGYDL'), ('HEQZOVEQNELEVEAYWCPI', 'IPCWYAEVELENQEVOZQEH'), ('WBFVOIKCRCETXOVRDNRS', 'SRNDRVOXTECRCKIOVFBW'), ('YJVSLYJSXHZUJUMHKGGS', 'SGGKHMUJUZHXSJYLSVJY'), ('SWCRQLEGCDZCOPOLGYHC', 'CHYGLOPOCZDCGELQRCWS'), ('TWQETASXSXJSXYFOQKDG', 'GDKQOFYXSJXSXSATEQWT'), ('WLMKVHVSJTZBSLOTFKBP', 'PBKFTOLSBZTJSVHVKMLW'), ('